# Time-Series Classification with a Transformer (FordA)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/river-kris/Keras-Timeseries/blob/main/notebooks/01_transformer_classification.ipynb)

Based on the official Keras example
[Timeseries classification with a Transformer model](https://keras.io/examples/timeseries/timeseries_classification_transformer/)
(author: Theodoros Ntakouris). Model and training code are kept identical to the official example;
this notebook only adds a `CONFIG` block, seeding, timing, and saving of results.

**How to use**
1. `Runtime > Change runtime type > T4 GPU`.
2. Edit `CONFIG` / `RUN_NAME` below (leave defaults for the baseline).
3. `Runtime > Run all`.
4. The last cell downloads `<RUN_NAME>.zip`. Unzip it into the repo's `results/` folder and commit.

## 1. Setup

In [ ]:
import os, json, time, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers

print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__, "| Python:", platform.python_version())
gpus = tf.config.list_physical_devices("GPU")
print("GPU:", gpus if gpus else "NONE - switch to Runtime > Change runtime type > T4 GPU")

In [ ]:
def make_run_dir(run_name):
    run_dir = os.path.join("outputs", run_name)
    os.makedirs(run_dir, exist_ok=True)
    return run_dir


def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=float)


def env_info():
    gpus = tf.config.list_physical_devices("GPU")
    gpu_name = "none"
    if gpus:
        try:
            gpu_name = tf.config.experimental.get_device_details(gpus[0]).get("device_name", "gpu")
        except Exception:
            gpu_name = "gpu"
    return {"tensorflow": tf.__version__, "keras": keras.__version__, "gpu": gpu_name}


def plot_history(history_df, keys, title, path):
    plt.figure(figsize=(7, 4))
    for k in keys:
        if k in history_df:
            plt.plot(history_df["epoch"] + 1, history_df[k], label=k)
    plt.xlabel("epoch"); plt.title(title); plt.legend(); plt.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(path, dpi=150); plt.show()

## 2. Configuration
Defaults = official baseline. For an experiment, change one thing and give it a new `RUN_NAME`
(e.g. `transformer_heads2`, `transformer_blocks2`).

In [ ]:
RUN_NAME = "transformer_baseline"

CONFIG = {
    "seed": 42,
    "head_size": 256,
    "num_heads": 4,
    "ff_dim": 4,
    "num_transformer_blocks": 4,
    "mlp_units": [128],
    "dropout": 0.25,
    "mlp_dropout": 0.4,
    "learning_rate": 1e-4,
    "batch_size": 64,
    "epochs": 150,
    "early_stopping_patience": 10,
    "validation_split": 0.2,
}

keras.utils.set_random_seed(CONFIG["seed"])
run_dir = make_run_dir(RUN_NAME)
print("Saving outputs to", run_dir)

## 3. Data: FordA
Each sample is 500 time steps of engine noise from one sensor; the label says whether the engine
has a fault. Labels are -1/1 in the raw data and get mapped to 0/1.

In [ ]:
def readucr(filename):
    data = np.loadtxt(filename, delimiter="\t")
    y = data[:, 0]
    x = data[:, 1:]
    return x, y.astype(int)


root_url = "https://raw.githubusercontent.com/hfawaz/cd-diagram/master/FordA/"

x_train, y_train = readucr(root_url + "FordA_TRAIN.tsv")
x_test, y_test = readucr(root_url + "FordA_TEST.tsv")

x_train = x_train.reshape((x_train.shape[0], x_train.shape[1], 1))
x_test = x_test.reshape((x_test.shape[0], x_test.shape[1], 1))

n_classes = len(np.unique(y_train))

idx = np.random.permutation(len(x_train))
x_train = x_train[idx]
y_train = y_train[idx]

y_train[y_train == -1] = 0
y_test[y_test == -1] = 0

print("x_train:", x_train.shape, "x_test:", x_test.shape, "classes:", n_classes)
print("train class counts:", np.bincount(y_train), "test class counts:", np.bincount(y_test))

In [ ]:
# One example of each class
plt.figure(figsize=(8, 3))
for c in np.unique(y_train):
    plt.plot(x_train[y_train == c][0, :, 0], label=f"class {c}")
plt.legend(); plt.title("FordA: one sample per class"); plt.tight_layout()
plt.savefig(os.path.join(run_dir, "data_samples.png"), dpi=150); plt.show()

## 4. Model

In [ ]:
def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0):
    # Attention and Normalization
    x = layers.MultiHeadAttention(
        key_dim=head_size, num_heads=num_heads, dropout=dropout
    )(inputs, inputs)
    x = layers.Dropout(dropout)(x)
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    res = x + inputs

    # Feed Forward Part
    x = layers.Conv1D(filters=ff_dim, kernel_size=1, activation="relu")(res)
    x = layers.Dropout(dropout)(x)
    x = layers.Conv1D(filters=inputs.shape[-1], kernel_size=1)(x)
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    return x + res


def build_model(
    input_shape,
    head_size,
    num_heads,
    ff_dim,
    num_transformer_blocks,
    mlp_units,
    dropout=0,
    mlp_dropout=0,
):
    inputs = keras.Input(shape=input_shape)
    x = inputs
    for _ in range(num_transformer_blocks):
        x = transformer_encoder(x, head_size, num_heads, ff_dim, dropout)

    x = layers.GlobalAveragePooling1D(data_format="channels_last")(x)
    for dim in mlp_units:
        x = layers.Dense(dim, activation="relu")(x)
        x = layers.Dropout(mlp_dropout)(x)
    outputs = layers.Dense(n_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs)


model = build_model(
    x_train.shape[1:],
    head_size=CONFIG["head_size"],
    num_heads=CONFIG["num_heads"],
    ff_dim=CONFIG["ff_dim"],
    num_transformer_blocks=CONFIG["num_transformer_blocks"],
    mlp_units=CONFIG["mlp_units"],
    mlp_dropout=CONFIG["mlp_dropout"],
    dropout=CONFIG["dropout"],
)
model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=keras.optimizers.Adam(learning_rate=CONFIG["learning_rate"]),
    metrics=["sparse_categorical_accuracy"],
)
model.summary()

## 5. Train

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        patience=CONFIG["early_stopping_patience"], restore_best_weights=True
    ),
    keras.callbacks.CSVLogger(os.path.join(run_dir, "history.csv")),
]

t0 = time.time()
history = model.fit(
    x_train,
    y_train,
    validation_split=CONFIG["validation_split"],
    epochs=CONFIG["epochs"],
    batch_size=CONFIG["batch_size"],
    callbacks=callbacks,
)
train_time_s = time.time() - t0
print(f"Training time: {train_time_s/60:.1f} min")

## 6. Evaluate

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=1)

hist = pd.DataFrame(history.history)
hist["epoch"] = np.arange(len(hist))
best_epoch = int(hist["val_loss"].idxmin())

y_pred = model.predict(x_test, verbose=0).argmax(axis=1)
cm = tf.math.confusion_matrix(y_test, y_pred, num_classes=n_classes).numpy()

metrics = {
    "run_name": RUN_NAME,
    "task": "classification",
    "model": "transformer",
    "test_accuracy": float(test_acc),
    "test_loss": float(test_loss),
    "best_val_accuracy": float(hist.loc[best_epoch, "val_sparse_categorical_accuracy"]),
    "best_val_loss": float(hist.loc[best_epoch, "val_loss"]),
    "best_epoch": best_epoch + 1,
    "epochs_run": int(len(hist)),
    "params": int(model.count_params()),
    "train_time_min": round(train_time_s / 60, 2),
    "confusion_matrix": cm.tolist(),
    **env_info(),
}
print(json.dumps(metrics, indent=2))

## 7. Save results

In [ ]:
save_json(CONFIG, os.path.join(run_dir, "config.json"))
save_json(metrics, os.path.join(run_dir, "metrics.json"))
model.save_weights(os.path.join(run_dir, "model.weights.h5"))
with open(os.path.join(run_dir, "model_summary.txt"), "w") as f:
    model.summary(print_fn=lambda s, **kw: f.write(s + "\n"))

plot_history(hist, ["loss", "val_loss"], f"{RUN_NAME}: loss", os.path.join(run_dir, "loss.png"))
plot_history(
    hist,
    ["sparse_categorical_accuracy", "val_sparse_categorical_accuracy"],
    f"{RUN_NAME}: accuracy",
    os.path.join(run_dir, "accuracy.png"),
)
print(os.listdir(run_dir))

In [ ]:
# Zip this run's outputs and download them.
# Unzip into the repo so the files end up at results/<RUN_NAME>/...
import shutil
archive = shutil.make_archive(RUN_NAME, "zip", root_dir="outputs", base_dir=RUN_NAME)
print("Created", archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("Not running in Colab - copy", run_dir, "into results/ manually.")

## 8. Observations
_Write what you noticed: how many epochs before early stopping, train vs. validation gap,
final test accuracy vs. the ~85% reported in the official example, training time._